# Global Forest Analytics Pipeline

This notebook builds a complete forest analytics pipeline from raw country-level forest datasets.

## Main outputs

The pipeline produces three structured analytical assets:

1. **canonical_country_table**  
   A master table with one row per country and baseline forest indicators.

2. **country_year_feature_store**  
   A country-by-year analytical table containing annual loss, primary loss, emissions, rolling averages, lags, and year-over-year change metrics.

3. **region_aggregates**  
   A hierarchical summary layer aggregating performance across:
   - World
   - Continent
   - Subregion

## Why this notebook matters

The purpose of this notebook is to transform raw forest data into a reusable modelling and analytics foundation that can support:

- dashboards
- filtering systems
- rankings
- time-series trend analysis
- anomaly detection
- forecasting
- regional benchmarking

## Pipeline flow

The notebook is organized into the following sections:

1. Imports and configuration  
2. Geography mapping  
3. Data loading  
4. Canonical country table construction  
5. Country-year feature store construction  
6. Regional aggregation layer  
7. Build outputs  
8. Validation and inspection

## Dataset root

This notebook uses the following dataset root:

`/Global Forest Analysis/dataset`

# Step 0: IMPORT LIBRARIES

In [26]:
# =========================================================
# STEP 1: IMPORTS AND CONFIGURATION
# =========================================================

from __future__ import annotations

# Standard library
from pathlib import Path
from urllib import request
from typing import Dict, Optional, List
from datetime import datetime
import warnings

# Data manipulation and analysis
from scipy.stats import linregress

# Data handling
import pandas as pd
import numpy as np

# For caching downloaded data
import hashlib
import json
import tempfile
import shutil

# YAML support for subregion mapping
import yaml

# Date handling
import datetime

# Plotting
import matplotlib

# Use Mac-compatible backend
matplotlib.use("TkAgg")

import matplotlib.pyplot as plt

# Optional ISO -> continent support
try:
    import pycountry_convert as pc
except ImportError:
    pc = None
    print("Warning: pycountry_convert is not installed. Continent mapping may return 'Unknown'.")

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# DATASET ROOT
# ---------------------------------------------------------
# This is the root folder containing all source CSV files.
DATASET_ROOT = Path("dataset")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset folder not found at:\n{DATASET_ROOT}\n\n"
        "Please confirm the folder path before running the rest of the notebook."
    )

print("Dataset root:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())

Dataset root: dataset
Dataset exists: True


In [20]:
# -----------------------------
# EDA
# -----------------------------

# 1️⃣ Set dataset path
iso_dataset_path = Path("dataset/iso_metadata.csv")


# 2️⃣ Download file if it doesn't exist
if not iso_dataset_path.exists():
   print("Downloading iso_metadata.csv...")
   iso_dataset_path.parent.mkdir(parents=True, exist_ok=True) 
   url = "..."
   try:
       response = request.urlopen(url)
       with open(iso_dataset_path, 'w', encoding='utf-8') as f:
           f.write(response.read().decode('utf-8'))
       print("✅ Download complete.")
   except request.URLError as e:
       print(f"❌ Error downloading dataset: {e}")
else:
   print("✅ Dataset already exists. Loading from file.")


# 3️⃣ Load dataset into pandas DataFrame
# Using comma as separator (default for this file)
iso_dataset_items = pd.read_csv(iso_dataset_path)


# 4️⃣ Inspect the first and last 5 rows
print("First 5 rows:")
print(iso_dataset_items.head())


print("\nLast 5 rows:")
iso_dataset_items.tail()

# -----------------------------
# Continent mapping
# -----------------------------


CONTINENT_CODE_TO_NAME = {
   "AF": "Africa",
   "AS": "Asia",
   "EU": "Europe",
   "NA": "North America",
   "SA": "South America",
   "OC": "Oceania",
   "AN": "Antarctica",
}

SPECIAL_ISO3_TO_ISO2 = {"XAD", "XCA", "XCL", "XKO", "XPI", "XSP", "XSX"}


def iso3_to_continent(iso3: str) -> str:
   """Convert ISO3 → continent name."""
   if not iso3:
       return "Unknown"


   iso3 = iso3.upper().strip()


   if iso3 in SPECIAL_ISO3_TO_ISO2:
       return "Unknown"


   try:
       iso2 = pc.country_alpha3_to_country_alpha2(iso3)
       continent_code = pc.country_alpha2_to_continent_code(iso2)
       return CONTINENT_CODE_TO_NAME.get(continent_code, "Unknown")
   except Exception:
       return "Unknown"


# -----------------------------
# Geography Mapper (Helper class to add continent and subregion metadata)
# -----------------------------


# ---------------------------------------------------------------------
# ISO3 -> Continent mapping
# ---------------------------------------------------------------------
# NOTE:
# This should already exist in your notebook/project. If you already have
# an `iso3_to_continent` dictionary defined elsewhere, keep that version
# and remove this placeholder.
#
# Example:
# iso3_to_continent = {
#     "KEN": "Africa",
#     "UGA": "Africa",
#     "TZA": "Africa",
#     ...
# }
# ---------------------------------------------------------------------
iso3_to_continent = globals().get("iso3_to_continent", {})


class GeographyMapper:
   """
   Adds continent and subregion metadata to country-level dataframes.


   Parameters
   ----------
   subregion_config_path : Optional[str | Path]
       Path to a YAML file containing custom subregion mappings.


   Expected YAML structure
   -----------------------
   subregions:
     Africa:
       Eastern Africa: [KEN, UGA, TZA, RWA, BDI, ETH, SOM, SSD, ERI, DJI, COM]
       Western Africa: [NGA, GHA, CIV]
     Europe:
       Northern Europe: [SWE, NOR, FIN]
   """

   def __init__(self, subregion_config_path: Optional[str | Path] = None):
       self.subregion_map: Dict[str, Dict[str, str]] = {}


       if subregion_config_path:
           self.subregion_map = self._load_subregion_map(Path(subregion_config_path))


   @staticmethod
   def _load_subregion_map(path: Path) -> Dict[str, Dict[str, str]]:
       """
       Load a YAML-based continent -> iso -> subregion mapping.


       Returns
       -------
       Dict[str, Dict[str, str]]
           Example:
           {
               "Africa": {
                   "KEN": "Eastern Africa",
                   "UGA": "Eastern Africa"
               }
           }
       """
       if not path.exists():
           print(f"[GeographyMapper] Warning: subregion config not found at {path}. Using empty mapping.")
           return {}


       payload = yaml.safe_load(path.read_text()) or {}
       subregions = payload.get("subregions", {})
       mapping: Dict[str, Dict[str, str]] = {}


       for continent, region_dict in subregions.items():
           mapping[continent] = {}


           if not isinstance(region_dict, dict):
               continue


           for subregion_name, iso_list in region_dict.items():
               for iso in iso_list or []:
                   mapping[continent][str(iso).upper()] = subregion_name


       return mapping


   def add_geography(self, df: pd.DataFrame, iso_col: str = "iso") -> pd.DataFrame:
       """
       Add continent and subregion columns to a dataframe.


       Important
       ---------
       This method preserves all existing columns in the dataframe.


       Parameters
       ----------
       df : pd.DataFrame
           Input dataframe containing an ISO3 country code column.
       iso_col : str, default="iso"
           Name of the ISO3 column.


       Returns
       -------
       pd.DataFrame
           Original dataframe plus:
           - continent
           - subregion
       """
       out = df.copy()


       if iso_col not in out.columns:
           raise KeyError(
               f"'{iso_col}' column not found in dataframe. "
               f"Available columns: {list(out.columns)}"
           )


       out[iso_col] = out[iso_col].astype(str).str.upper().str.strip()


       # Add continent if it is missing
       if "continent" not in out.columns:
           out["continent"] = out[iso_col].map(iso3_to_continent).fillna("Unknown")


       # Add subregion from YAML map if available
       def map_subregion(row) -> str:
           continent = row.get("continent", "Unknown")
           iso = str(row[iso_col]).upper()
           return self.subregion_map.get(continent, {}).get(iso, "Unassigned")


       if "subregion" not in out.columns:
           out["subregion"] = out.apply(map_subregion, axis=1)
       else:
           out["subregion"] = out["subregion"].fillna(
               out.apply(map_subregion, axis=1)
           )


       return out
# -----------------------------
# Hashing and logging utilities
# -----------------------------


def hash_dataframe(df: pd.DataFrame) -> str:
   """Create deterministic hash of dataframe."""
   df_sorted = df.sort_values(list(df.columns)).reset_index(drop=True)
   return hashlib.sha256(df_sorted.to_csv(index=False).encode()).hexdigest()


def safe_write_csv(df: pd.DataFrame, path: Path):
   """Atomic CSV write to prevent corruption."""
   with tempfile.NamedTemporaryFile(delete=False, suffix=".csv") as tmp:
       tmp_path = Path(tmp.name)


   df.to_csv(tmp_path, index=False)
   shutil.move(tmp_path, path)


def log_dataset_version(log_file: Path, record: dict):
   """Append dataset update record."""
   try:
       with open(log_file, "a", encoding="utf-8") as f:
           f.write(json.dumps(record) + "\n")
   except Exception as e:
       print(f"⚠️ logging failed: {e}")


def save_country_geography(df: pd.DataFrame):


   # Ensure schema compatibility with pipeline
   required_cols = ["name", "iso", "continent", "subregion"]
   missing = [c for c in required_cols if c not in df.columns]


   if missing:
       raise ValueError(f"Missing required columns: {missing}")


   df = df[required_cols].copy()


   folder = Path("dataset")
   file_path = folder / "country_geography.csv"
   hash_file = folder / "country_geography.hash"
   log_file = folder / "dataset_versions.log"


   try:


       # -----------------------------
       # Folder creation
       # -----------------------------
       if not folder.exists():
           folder.mkdir(parents=True, exist_ok=True)


           df_hash = hash_dataframe(df)


           safe_write_csv(df, file_path)
           hash_file.write_text(df_hash)


           log_dataset_version(
               log_file,
               {
                   "dataset": "country_geography",
                   "timestamp": datetime.utcnow().isoformat(),
                   "rows": len(df),
                   "hash": df_hash,
                   "action": "created"
               },
           )


           print('✅ new folder created, "dataset" file "country_geography.csv" saved successfully')
           return


       # -----------------------------
       # File does not exist
       # -----------------------------
       if not file_path.exists():


           df_hash = hash_dataframe(df)


           safe_write_csv(df, file_path)
           hash_file.write_text(df_hash)


           log_dataset_version(
               log_file,
               {
                   "dataset": "country_geography",
                   "timestamp": datetime.utcnow().isoformat(),
                   "rows": len(df),
                   "hash": df_hash,
                   "action": "created"
               },
           )


           print('✅ file "country_geography.csv" saved in dataset successfully')
           return


       # -----------------------------
       # Compare hashes
       # -----------------------------
       new_hash = hash_dataframe(df)
       old_hash = hash_file.read_text().strip() if hash_file.exists() else None


       if new_hash == old_hash:
           print("✅ file already saved")
           return


       # -----------------------------
       # Update dataset
       # -----------------------------
       safe_write_csv(df, file_path)
       hash_file.write_text(new_hash)


       log_dataset_version(
           log_file,
           {
               "dataset": "country_geography",
               "timestamp": datetime.utcnow().isoformat(),
               "rows": len(df),
               "hash": new_hash,
               "action": "updated"
           },
       )


       print('✅ file updated and saved in dataset successfully')


   except Exception as e:
       print(f"❌ dataset save failed: {e}")
      
# -----------------------------
# Save the resulting DataFrame to CSV
# -----------------------------
mapper = GeographyMapper("subregions.yaml")


geo_df = mapper.add_geography(iso_dataset_items)


save_country_geography(geo_df)


# -----------------------------
# Run
# -----------------------------
mapper = GeographyMapper("subregions.yaml")


geo_df = mapper.add_geography(iso_dataset_items)


# Check the first 5 rows of the resulting DataFrame
print("First 5 rows of the geographic mapping:")
print(geo_df.head())


# Check the last 5 rows of the resulting DataFrame
print("Last 5 rows of the geographic mapping:")
geo_df.tail()



✅ Dataset already exists. Loading from file.
First 5 rows:
                    name  iso
0            Afghanistan  AFG
1  Akrotiri and Dhekelia  XAD
2                  Åland  ALA
3                Albania  ALB
4                Algeria  DZA

Last 5 rows:
✅ file already saved
First 5 rows of the geographic mapping:
                    name  iso continent        subregion
0            Afghanistan  AFG      Asia    Southern Asia
1  Akrotiri and Dhekelia  XAD   Unknown       Unassigned
2                  Åland  ALA    Europe  Northern Europe
3                Albania  ALB    Europe  Southern Europe
4                Algeria  DZA    Africa  Northern Africa
Last 5 rows of the geographic mapping:


,name,iso,continent,subregion
242,"Virgin Islands, U.S.",VIR,North America,Caribbean
243,Wallis and Futuna,WLF,Oceania,Polynesia
244,Yemen,YEM,Asia,Western Asia
245,Zambia,ZMB,Africa,Eastern Africa
246,Zimbabwe,ZWE,Africa,Eastern Africa


# 1. Core Data Structures

## STEP 2: GEO MAPPING

This section defines the geography mapping layer used throughout the notebook.

### Purpose

The geography layer enriches country-level data with:

- `continent`
- `subregion`

This makes it possible to organize results according to the hierarchy:

**Global → Continent → Subregion → Country**

### Notes

- Continents are inferred from ISO3 country codes.
- Subregions are read from an optional YAML configuration.
- If no YAML file is provided, countries without a custom mapping are assigned `Unassigned`.
- The geography mapper always preserves existing columns and only adds new geography fields.

In [6]:
# =========================================================
# STEP 2: GEOGRAPHY MAPPING
# =========================================================

# ---------------------------------------------------------
# Continent code -> continent name
# ---------------------------------------------------------
CONTINENT_CODE_TO_NAME = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica",
}

# Some unusual or non-standard ISO3 values may not map cleanly
SPECIAL_ISO3_TO_ISO2 = {"XAD", "XCA", "XCL", "XKO", "XPI", "XSP", "XSX"}


def iso3_to_continent(iso3: str) -> str:
    """
    Convert an ISO3 country code to a continent name.

    Parameters
    ----------
    iso3 : str
        Three-letter ISO country code.

    Returns
    -------
    str
        Continent name if resolved, otherwise 'Unknown'.
    """
    if not iso3 or pc is None:
        return "Unknown"

    iso3 = str(iso3).upper().strip()

    if iso3 in SPECIAL_ISO3_TO_ISO2:
        return "Unknown"

    try:
        iso2 = pc.country_alpha3_to_country_alpha2(iso3)
        continent_code = pc.country_alpha2_to_continent_code(iso2)
        return CONTINENT_CODE_TO_NAME.get(continent_code, "Unknown")
    except Exception:
        return "Unknown"


class GeographyMapper:
    """
    Add continent and subregion information to a dataframe.

    Parameters
    ----------
    subregion_config_path : Optional[str | Path]
        Optional YAML file containing custom subregion definitions.
    """

    def __init__(self, subregion_config_path: Optional[str | Path] = None):
        self.subregion_map: Dict[str, Dict[str, str]] = {}

        if subregion_config_path:
            self.subregion_map = self._load_subregion_map(Path(subregion_config_path))

    @staticmethod
    def _load_subregion_map(path: Path) -> Dict[str, Dict[str, str]]:
        """
        Load subregion mappings from a YAML file.

        Expected YAML structure
        -----------------------
        subregions:
          Africa:
            Eastern Africa: [KEN, UGA, TZA, RWA]
            Western Africa: [NGA, GHA]
          Europe:
            Northern Europe: [SWE, NOR, FIN]
        """
        if not path.exists():
            print(f"[GeographyMapper] Warning: YAML config not found at {path}. Using empty subregion mapping.")
            return {}

        payload = yaml.safe_load(path.read_text()) or {}
        subregions = payload.get("subregions", {})

        mapping: Dict[str, Dict[str, str]] = {}

        for continent, region_dict in subregions.items():
            mapping[continent] = {}

            if not isinstance(region_dict, dict):
                continue

            for subregion_name, iso_list in region_dict.items():
                for iso in iso_list or []:
                    mapping[continent][str(iso).upper()] = subregion_name

        return mapping

    def add_geography(self, df: pd.DataFrame, iso_col: str = "iso") -> pd.DataFrame:
        """
        Add continent and subregion columns while preserving all existing columns.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe.
        iso_col : str, default='iso'
            Name of ISO3 column.

        Returns
        -------
        pd.DataFrame
            Original dataframe plus:
            - continent
            - subregion
        """
        out = df.copy()

        if iso_col not in out.columns:
            raise KeyError(
                f"'{iso_col}' column not found in dataframe. "
                f"Available columns: {list(out.columns)}"
            )

        out[iso_col] = out[iso_col].astype(str).str.upper().str.strip()

        if "continent" not in out.columns:
            out["continent"] = out[iso_col].map(iso3_to_continent).fillna("Unknown")

        def map_subregion(row) -> str:
            continent = row.get("continent", "Unknown")
            iso = str(row[iso_col]).upper()
            return self.subregion_map.get(continent, {}).get(iso, "Unassigned")

        if "subregion" not in out.columns:
            out["subregion"] = out.apply(map_subregion, axis=1)
        else:
            out["subregion"] = out["subregion"].fillna(out.apply(map_subregion, axis=1))

        return out

## STEP 3: DATA LOADING LAYER 

This section loads all required source tables from the dataset folder.

### Expected source tables

The notebook expects the following logical inputs:

- ISO metadata
- forest extent
- forest gain
- net forest change
- annual forest loss
- annual primary forest loss
- deforestation rank

### Output

All loaded datasets are stored in a dictionary called `tables`.

This dictionary is the standard input passed into the feature engineering pipeline.

In [7]:
# =========================================================
# STEP 3: DATA LOADING
# =========================================================

def _read_csv(path: Path) -> pd.DataFrame:
    """
    Read a CSV file with validation.
    """
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)


def load_raw_tables(data_root: str | Path) -> Dict[str, pd.DataFrame]:
    """
    Load all required raw tables.

    Parameters
    ----------
    data_root : str | Path
        Root dataset folder.

    Returns
    -------
    Dict[str, pd.DataFrame]
        Dictionary containing named data tables.
    """
    data_root = Path(data_root)

    def_path = data_root / "LP" / "fao_treecover_deforestation_rank.csv"
    if not def_path.exists():
        def_path = data_root / "MP" / "fao_treecover_deforestation_rank.csv"

    tables = {
        "iso_meta": _read_csv(data_root / "country_geography.csv"),
        "extent": _read_csv(data_root / "HP" / "treecover_extent_2000_by_region__ha.csv"),
        "gain": _read_csv(data_root / "HP" / "treecover_gain_2000-2020_by_region__ha.csv"),
        "net_change": _read_csv(data_root / "HP" / "net_tree_cover_change_from_height__ha.csv"),
        "loss_annual": _read_csv(data_root / "HP" / "treecover_loss_by_region__ha.csv"),
        "primary_loss_annual": _read_csv(data_root / "HP" / "treecover_loss_in_primary_forests_2001-2020_by_region__ha.csv"),
        "deforestation_rank": _read_csv(def_path),
    }

    return tables


# Load all source tables now so the rest of the notebook can use them
tables = load_raw_tables(DATASET_ROOT)

print("Loaded tables:")
for name, df in tables.items():
    print(f"- {name}: {df.shape}")
    
    

Loaded tables:
- iso_meta: (247, 4)
- extent: (235, 3)
- gain: (219, 2)
- net_change: (256, 8)
- loss_annual: (4776, 4)
- primary_loss_annual: (2225, 4)
- deforestation_rank: (123, 3)


## STEP 4: FOREST FEATURE ENGINEERING

This section defines the core feature builder.

### Outputs produced here

The `ForestFeatureBuilder` creates:

1. **canonical_country_table**  
   A country-level master table with long-horizon and baseline metrics.

2. **country_year_feature_store**  
   A country-year analytical table with annual measures and derived time-series features.

### Design goals

The implementation is designed to:

- validate required tables and columns
- standardize column names safely
- preserve all geography columns
- avoid repetitive merge logic
- compute all major derived metrics in one place
- support downstream aggregation and modelling

In [24]:
# =========================================================
# STEP 4: FOREST FEATURE BUILDER
# =========================================================

class ForestFeatureBuilder:
    """
    Build country-level and country-year forest analytics assets.
    """

    def __init__(self, geography_mapper: GeographyMapper):
        self.geography_mapper = geography_mapper

    # -----------------------------------------------------
    # Validation helpers
    # -----------------------------------------------------
    @staticmethod
    def _require_tables(tables: Dict[str, pd.DataFrame], required_tables: List[str]) -> None:
        missing = [t for t in required_tables if t not in tables]
        if missing:
            raise KeyError(
                f"Missing required table(s): {missing}. "
                f"Available tables: {list(tables.keys())}"
            )

    @staticmethod
    def _require_columns(df: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(
                f"Missing required column(s) {missing} in table '{table_name}'. "
                f"Available columns: {list(df.columns)}"
            )

    @staticmethod
    def _safe_rename(df: pd.DataFrame, rename_map: Dict[str, str]) -> pd.DataFrame:
        """
        Rename only columns that are present.
        """
        applicable = {k: v for k, v in rename_map.items() if k in df.columns}
        return df.rename(columns=applicable)

    @staticmethod
    def _ensure_columns(df: pd.DataFrame, columns_with_defaults: Dict[str, object]) -> pd.DataFrame:
        """
        Ensure expected columns exist, creating them if necessary.
        """
        out = df.copy()
        for col, default in columns_with_defaults.items():
            if col not in out.columns:
                out[col] = default
        return out

    # -----------------------------------------------------
    # Canonical country table
    # -----------------------------------------------------
    def build_canonical_country_table(self, tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Build a canonical country table with one row per country.

        The resulting table acts as the master country-level dataset and
        includes baseline indicators and derived features needed for
        analytics and modelling.
        """
        self._require_tables(
            tables,
            ["iso_meta", "extent", "gain", "loss_annual", "primary_loss_annual"]
        )

        # -----------------------------
        # 1. ISO metadata
        # -----------------------------
        iso_meta = tables["iso_meta"].copy()
        self._require_columns(iso_meta, ["iso"], "iso_meta")

        if "name" in iso_meta.columns and "country_name" not in iso_meta.columns:
            iso_meta = iso_meta.rename(columns={"name": "country_name"})

        if "country_name" not in iso_meta.columns:
            iso_meta["country_name"] = iso_meta["iso"]

        iso_meta["iso"] = iso_meta["iso"].astype(str).str.upper().str.strip()

        # -----------------------------
        # 2. Extent
        # -----------------------------
        extent = tables["extent"].copy()
        self._require_columns(extent, ["iso"], "extent")
        extent["iso"] = extent["iso"].astype(str).str.upper().str.strip()

        extent = self._safe_rename(
            extent,
            {
                "umd_tree_cover_extent_2000__ha": "extent_2000_ha",
                "extent_2000": "extent_2000_ha",
                "tree_cover_extent_2000_ha": "extent_2000_ha",
                "area": "area_ha",
                "country_area_ha": "area_ha",
            }
        )

        # -----------------------------
        # 3. Gain
        # -----------------------------
        gain = tables["gain"].copy()
        self._require_columns(gain, ["iso"], "gain")
        gain["iso"] = gain["iso"].astype(str).str.upper().str.strip()

        gain = self._safe_rename(
            gain,
            {
                "umd_tree_cover_gain_2000_2020__ha": "gain_2000_2020_ha",
                "gain_2000_2020": "gain_2000_2020_ha",
                "tree_cover_gain_2000_2020_ha": "gain_2000_2020_ha",
            }
        )

        # -----------------------------
        # 4. Annual loss
        # -----------------------------
        loss_annual = tables["loss_annual"].copy()
        self._require_columns(loss_annual, ["iso"], "loss_annual")
        loss_annual["iso"] = loss_annual["iso"].astype(str).str.upper().str.strip()

        loss_annual = self._safe_rename(
            loss_annual,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_emissions_Mg",
            }
        )

        self._require_columns(loss_annual, ["iso", "year", "annual_loss_ha"], "loss_annual")

        loss_agg = (
            loss_annual.groupby("iso", as_index=False)
            .agg(
                loss_total_2001_2020=("annual_loss_ha", "sum"),
                avg_annual_loss_ha=("annual_loss_ha", "mean"),
            )
        )

        # -----------------------------
        # 5. Annual primary loss
        # -----------------------------
        primary_loss_annual = tables["primary_loss_annual"].copy()
        self._require_columns(primary_loss_annual, ["iso"], "primary_loss_annual")
        primary_loss_annual["iso"] = primary_loss_annual["iso"].astype(str).str.upper().str.strip()

        primary_loss_annual = self._safe_rename(
            primary_loss_annual,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_primary_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_primary_emissions_Mg",
            }
        )

        self._require_columns(
            primary_loss_annual,
            ["iso", "year", "annual_primary_loss_ha"],
            "primary_loss_annual"
        )

        primary_loss_agg = (
            primary_loss_annual.groupby("iso", as_index=False)
            .agg(
                primary_loss_total_2001_2020=("annual_primary_loss_ha", "sum"),
            )
        )

        # -----------------------------
        # 6. Merge core static components
        # -----------------------------
        canonical = (
            iso_meta
            .merge(extent.drop(columns=["name"], errors="ignore"), on="iso", how="left")
            .merge(gain.drop(columns=["name"], errors="ignore"), on="iso", how="left")
            .merge(loss_agg, on="iso", how="left")
            .merge(primary_loss_agg, on="iso", how="left")
        )

        # Add continent and subregion without dropping existing columns
        canonical = self.geography_mapper.add_geography(canonical)

        # -----------------------------
        # 7. Ensure expected columns exist
        # -----------------------------
        canonical = self._ensure_columns(
            canonical,
            {
                "country_name": pd.NA,
                "area_ha": pd.NA,
                "extent_2000_ha": pd.NA,
                "gain_2000_2020_ha": 0.0,
                "loss_total_2001_2020": 0.0,
                "primary_loss_total_2001_2020": 0.0,
                "avg_annual_loss_ha": pd.NA,
            }
        )

        # -----------------------------
        # 8. Numeric cleanup
        # -----------------------------
        numeric_cols = [
            "area_ha",
            "extent_2000_ha",
            "gain_2000_2020_ha",
            "loss_total_2001_2020",
            "primary_loss_total_2001_2020",
            "avg_annual_loss_ha",
        ]

        for col in numeric_cols:
            canonical[col] = pd.to_numeric(canonical[col], errors="coerce")

        # -----------------------------
        # 9. Derived metrics
        # -----------------------------
        canonical["net_change_ha"] = (
            canonical["gain_2000_2020_ha"].fillna(0) -
            canonical["loss_total_2001_2020"].fillna(0)
        )

        canonical["net_change_pct"] = (
            canonical["net_change_ha"] /
            canonical["extent_2000_ha"].replace(0, np.nan)
        ) * 100

        canonical["recovery_gap_ha"] = (
            canonical["gain_2000_2020_ha"].fillna(0) -
            canonical["loss_total_2001_2020"].fillna(0)
        )

        canonical["reforestation_rate"] = (
            canonical["gain_2000_2020_ha"] /
            canonical["loss_total_2001_2020"].replace(0, np.nan)
        )

        canonical["loss_per_area"] = (
            canonical["loss_total_2001_2020"] /
            canonical["area_ha"].replace(0, np.nan)
        )

        if "disturbance_ha" not in canonical.columns:
            canonical["disturbance_ha"] = pd.NA

        canonical["deforestation_rank_value"] = canonical["loss_total_2001_2020"].rank(
            method="dense",
            ascending=False
        )

        canonical = canonical.drop_duplicates(subset=["iso"]).reset_index(drop=True)

        return canonical

    # -----------------------------------------------------
    # Country-year feature store
    # -----------------------------------------------------
    def build_country_year_feature_store(self, tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Build a country-year feature store containing annual measures and
        time-series features for each country.
        """
        self._require_tables(
            tables,
            ["loss_annual", "primary_loss_annual", "iso_meta", "extent", "gain"]
        )

        # -----------------------------
        # 1. Annual loss table
        # -----------------------------
        loss = tables["loss_annual"].copy()
        loss = self._safe_rename(
            loss,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_emissions_Mg",
            }
        )

        self._require_columns(loss, ["iso", "year", "annual_loss_ha"], "loss_annual")

        loss["iso"] = loss["iso"].astype(str).str.upper().str.strip()
        loss["year"] = pd.to_numeric(loss["year"], errors="coerce")

        if "annual_emissions_Mg" not in loss.columns:
            loss["annual_emissions_Mg"] = pd.NA

        # -----------------------------
        # 2. Annual primary loss table
        # -----------------------------
        primary = tables["primary_loss_annual"].copy()
        primary = self._safe_rename(
            primary,
            {
                "umd_tree_cover_loss__year": "year",
                "umd_tree_cover_loss__ha": "annual_primary_loss_ha",
                "gfw_gross_emissions_co2e_all_gases__Mg": "annual_primary_emissions_Mg",
            }
        )

        self._require_columns(primary, ["iso", "year", "annual_primary_loss_ha"], "primary_loss_annual")

        primary["iso"] = primary["iso"].astype(str).str.upper().str.strip()
        primary["year"] = pd.to_numeric(primary["year"], errors="coerce")

        if "annual_primary_emissions_Mg" not in primary.columns:
            primary["annual_primary_emissions_Mg"] = pd.NA

        # -----------------------------
        # 3. Country-level static features
        # -----------------------------
        country = self.build_canonical_country_table(tables)

        # -----------------------------
        # 4. Merge annual data
        # -----------------------------
        out = loss.merge(primary, on=["iso", "year"], how="left", suffixes=("", "_dup"))

        # -----------------------------
        # 5. Add static metadata
        # -----------------------------
        static_cols = [
            "iso",
            "country_name",
            "continent",
            "subregion",
            "area_ha",
            "extent_2000_ha",
            "gain_2000_2020_ha",
            "loss_total_2001_2020",
            "primary_loss_total_2001_2020",
            "avg_annual_loss_ha",
            "loss_per_area",
            "net_change_ha",
            "net_change_pct",
            "recovery_gap_ha",
            "disturbance_ha",
            "reforestation_rate",
            "deforestation_rank_value",
        ]

        available_static_cols = [c for c in static_cols if c in country.columns]

        out = (
            out.merge(country[available_static_cols], on="iso", how="left")
               .sort_values(["iso", "year"])
               .reset_index(drop=True)
        )

        # -----------------------------
        # 6. Numeric cleanup
        # -----------------------------
        for col in [
            "annual_loss_ha",
            "annual_emissions_Mg",
            "annual_primary_loss_ha",
            "annual_primary_emissions_Mg",
        ]:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce")

        # -----------------------------
        # 7. Time-series features by country
        # -----------------------------
        grp = out.groupby("iso", group_keys=False)

        # Lag features
        out["annual_loss_ha_lag1"] = grp["annual_loss_ha"].shift(1)
        out["annual_loss_ha_lag2"] = grp["annual_loss_ha"].shift(2)
        out["annual_primary_loss_ha_lag1"] = grp["annual_primary_loss_ha"].shift(1)
        out["annual_emissions_Mg_lag1"] = grp["annual_emissions_Mg"].shift(1)

        # Rolling features
        out["annual_loss_ha_roll3"] = grp["annual_loss_ha"].transform(
            lambda s: s.rolling(window=3, min_periods=1).mean()
        )
        out["annual_loss_ha_roll5"] = grp["annual_loss_ha"].transform(
            lambda s: s.rolling(window=5, min_periods=1).mean()
        )
        out["annual_primary_loss_ha_roll3"] = grp["annual_primary_loss_ha"].transform(
            lambda s: s.rolling(window=3, min_periods=1).mean()
        )
        out["annual_emissions_Mg_roll3"] = grp["annual_emissions_Mg"].transform(
            lambda s: s.rolling(window=3, min_periods=1).mean()
        )

        # Year-over-year changes
        out["annual_loss_ha_yoy_change"] = grp["annual_loss_ha"].diff()
        out["annual_primary_loss_ha_yoy_change"] = grp["annual_primary_loss_ha"].diff()

        # Growth rates
        out["annual_loss_ha_growth_rate"] = grp["annual_loss_ha"].pct_change()
        out["annual_primary_loss_ha_growth_rate"] = grp["annual_primary_loss_ha"].pct_change()

        # Loss relative to historical extent
        if "extent_2000_ha" in out.columns:
            out["annual_loss_pct_of_extent_2000"] = (
                out["annual_loss_ha"] / out["extent_2000_ha"].replace(0, np.nan)
            ) * 100

            out["annual_primary_loss_pct_of_extent_2000"] = (
                out["annual_primary_loss_ha"] / out["extent_2000_ha"].replace(0, np.nan)
            ) * 100

        return out

## STEP 5: REGIONAL AGGREGATION BUILDER

### Regional Aggregation Layer

This section defines the regional aggregation builder.

#### Purpose

The region aggregation layer summarizes annual country performance at three levels:

- **World**
- **Continent**
- **Subregion**

#### Why this is useful

This layer supports comparisons such as:

- Eastern Africa vs Africa
- Africa vs Global
- Asia vs Europe
- subregions against global baselines over time

#### Output structure

Each row in the aggregated output represents:

- one year
- one aggregation level
- one region name

In [9]:
# =========================================================
# STEP 5: REGIONAL AGGREGATION BUILDER
# =========================================================

class RegionalAggregationBuilder:
    """
    Build World-, Continent-, and Subregion-level aggregates from the
    country-year feature store.
    """

    def __init__(self, geography_mapper: GeographyMapper):
        self.geography_mapper = geography_mapper

    @staticmethod
    def _require_columns(df: pd.DataFrame, required_cols: List[str], df_name: str) -> None:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(
                f"Missing required column(s) {missing} in '{df_name}'. "
                f"Available columns: {list(df.columns)}"
            )

    @staticmethod
    def _safe_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
        out = df.copy()
        for col in cols:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce")
        return out

    @staticmethod
    def _add_group_timeseries_features(
        df: pd.DataFrame,
        value_cols: List[str],
        group_cols: List[str]
    ) -> pd.DataFrame:
        """
        Add lag, rolling average, YoY change, and growth features for each region.
        """
        out = df.copy()
        out = out.sort_values(group_cols + ["year"]).reset_index(drop=True)

        grp = out.groupby(group_cols, group_keys=False)

        for col in value_cols:
            if col not in out.columns:
                continue

            out[f"{col}_lag1"] = grp[col].shift(1)
            out[f"{col}_lag2"] = grp[col].shift(2)

            out[f"{col}_roll3"] = grp[col].transform(
                lambda s: s.rolling(window=3, min_periods=1).mean()
            )
            out[f"{col}_roll5"] = grp[col].transform(
                lambda s: s.rolling(window=5, min_periods=1).mean()
            )

            out[f"{col}_yoy_change"] = grp[col].diff()
            out[f"{col}_growth_rate"] = grp[col].pct_change()

        return out

    def _aggregate_one_level(
        self,
        df: pd.DataFrame,
        level_name: str,
        region_col: Optional[str]
    ) -> pd.DataFrame:
        """
        Aggregate one hierarchy level.
        """
        work = df.copy()

        annual_cols = [
            "annual_loss_ha",
            "annual_primary_loss_ha",
            "annual_emissions_Mg",
            "annual_primary_emissions_Mg",
        ]

        static_sum_cols = [
            "area_ha",
            "extent_2000_ha",
            "gain_2000_2020_ha",
            "loss_total_2001_2020",
            "primary_loss_total_2001_2020",
            "recovery_gap_ha",
            "disturbance_ha",
        ]

        static_mean_cols = [
            "avg_annual_loss_ha",
            "loss_per_area",
            "net_change_ha",
            "net_change_pct",
            "reforestation_rate",
            "deforestation_rank_value",
        ]

        for col in annual_cols + static_sum_cols + static_mean_cols:
            if col not in work.columns:
                work[col] = np.nan

        if level_name == "World":
            work["aggregation_level"] = "World"
            work["region_name"] = "Global"
        else:
            if region_col is None:
                raise ValueError(f"region_col cannot be None for level '{level_name}'")
            work["aggregation_level"] = level_name
            work["region_name"] = work[region_col].fillna("Unassigned")

        group_cols = ["aggregation_level", "region_name", "year"]

        out = (
            work.groupby(group_cols, dropna=False, as_index=False)
            .agg({
                "iso": pd.Series.nunique,
                "country_name": pd.Series.nunique if "country_name" in work.columns else "count",
                "annual_loss_ha": "sum",
                "annual_primary_loss_ha": "sum",
                "annual_emissions_Mg": "sum",
                "annual_primary_emissions_Mg": "sum",
                "area_ha": "sum",
                "extent_2000_ha": "sum",
                "gain_2000_2020_ha": "sum",
                "loss_total_2001_2020": "sum",
                "primary_loss_total_2001_2020": "sum",
                "recovery_gap_ha": "sum",
                "disturbance_ha": "sum",
                "avg_annual_loss_ha": "mean",
                "loss_per_area": "mean",
                "net_change_ha": "mean",
                "net_change_pct": "mean",
                "reforestation_rate": "mean",
                "deforestation_rank_value": "mean",
            })
            .rename(columns={"iso": "country_count", "country_name": "country_name_count"})
        )

        # Aggregate-level derived ratios
        out["annual_loss_pct_of_extent_2000"] = (
            out["annual_loss_ha"] / out["extent_2000_ha"].replace(0, np.nan)
        ) * 100

        out["annual_primary_loss_pct_of_extent_2000"] = (
            out["annual_primary_loss_ha"] / out["extent_2000_ha"].replace(0, np.nan)
        ) * 100

        out["aggregate_reforestation_rate"] = (
            out["gain_2000_2020_ha"] / out["loss_total_2001_2020"].replace(0, np.nan)
        )

        out["aggregate_net_change_ha"] = (
            out["gain_2000_2020_ha"] - out["loss_total_2001_2020"]
        )

        out["aggregate_net_change_pct"] = (
            out["aggregate_net_change_ha"] / out["extent_2000_ha"].replace(0, np.nan)
        ) * 100

        # Add time-series features at the regional level
        out = self._add_group_timeseries_features(
            out,
            value_cols=[
                "annual_loss_ha",
                "annual_primary_loss_ha",
                "annual_emissions_Mg",
                "annual_primary_emissions_Mg",
                "annual_loss_pct_of_extent_2000",
                "annual_primary_loss_pct_of_extent_2000",
            ],
            group_cols=["aggregation_level", "region_name"]
        )

        return out

    def build_region_aggregates(self, country_year_feature_store: pd.DataFrame) -> pd.DataFrame:
        """
        Build World, Continent, and Subregion annual aggregates.
        """
        df = country_year_feature_store.copy()

        self._require_columns(
            df,
            ["iso", "year", "continent", "subregion"],
            "country_year_feature_store"
        )

        df["iso"] = df["iso"].astype(str).str.upper().str.strip()
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df["continent"] = df["continent"].fillna("Unknown")
        df["subregion"] = df["subregion"].fillna("Unassigned")

        df = self._safe_numeric(
            df,
            [
                "annual_loss_ha",
                "annual_primary_loss_ha",
                "annual_emissions_Mg",
                "annual_primary_emissions_Mg",
                "area_ha",
                "extent_2000_ha",
                "gain_2000_2020_ha",
                "loss_total_2001_2020",
                "primary_loss_total_2001_2020",
                "avg_annual_loss_ha",
                "loss_per_area",
                "net_change_ha",
                "net_change_pct",
                "recovery_gap_ha",
                "disturbance_ha",
                "reforestation_rate",
                "deforestation_rank_value",
            ]
        )

        world_agg = self._aggregate_one_level(df, "World", None)
        continent_agg = self._aggregate_one_level(df, "Continent", "continent")
        subregion_agg = self._aggregate_one_level(df, "Subregion", "subregion")

        region_aggregates = pd.concat(
            [world_agg, continent_agg, subregion_agg],
            ignore_index=True
        )

        level_order = {"World": 0, "Continent": 1, "Subregion": 2}
        region_aggregates["aggregation_level_order"] = region_aggregates["aggregation_level"].map(level_order)

        region_aggregates = (
            region_aggregates
            .sort_values(["year", "aggregation_level_order", "region_name"])
            .drop(columns=["aggregation_level_order"])
            .reset_index(drop=True)
        )

        return region_aggregates

## STEP 6: BUILD OUTPUTS

### Build Pipeline Outputs

This section executes the full pipeline and creates the three main outputs:

- `canonical_country_table`
- `country_year_features`
- `region_aggregates`

In [10]:
# =========================================================
# STEP 6: BUILD OUTPUTS
# =========================================================

# Optional:
# If you have a YAML file for subregions, replace None with its path.
# Example:
# subregion_yaml_path = DATASET_ROOT / "subregions.yaml"
subregion_yaml_path = None

# Initialize builders
geo_mapper = GeographyMapper(subregion_config_path=subregion_yaml_path)
ffb = ForestFeatureBuilder(geography_mapper=geo_mapper)
rab = RegionalAggregationBuilder(geography_mapper=geo_mapper)

# Build outputs
canonical_country_table = ffb.build_canonical_country_table(tables)
country_year_features = ffb.build_country_year_feature_store(tables)
region_aggregates = rab.build_region_aggregates(country_year_features)

print("canonical_country_table shape:", canonical_country_table.shape)
print("country_year_features shape:", country_year_features.shape)
print("region_aggregates shape:", region_aggregates.shape)

canonical_country_table shape: (247, 19)
country_year_features shape: (4776, 36)
region_aggregates shape: (717, 63)


## STEP 7: VALIDATION AND INSPECTION
### Validation and Inspection

This section checks whether the pipeline outputs were built correctly.

The checks include:

- column inspection
- preview rows
- missing value checks
- geography quality checks

In [11]:
# =========================================================
# STEP 7: VALIDATION AND INSPECTION
# =========================================================

print("Canonical country table columns:")
print(canonical_country_table.columns.tolist())

print("\nCountry-year feature store columns:")
print(country_year_features.columns.tolist())

print("\nRegion aggregates columns:")
print(region_aggregates.columns.tolist())

Canonical country table columns:
['country_name', 'iso', 'continent', 'subregion', 'extent_2000_ha', 'area__ha', 'umd_tree_cover_gain__ha', 'loss_total_2001_2020', 'avg_annual_loss_ha', 'primary_loss_total_2001_2020', 'area_ha', 'gain_2000_2020_ha', 'net_change_ha', 'net_change_pct', 'recovery_gap_ha', 'reforestation_rate', 'loss_per_area', 'disturbance_ha', 'deforestation_rank_value']

Country-year feature store columns:
['iso', 'year', 'annual_loss_ha', 'annual_emissions_Mg', 'annual_primary_loss_ha', 'annual_primary_emissions_Mg', 'country_name', 'continent', 'subregion', 'area_ha', 'extent_2000_ha', 'gain_2000_2020_ha', 'loss_total_2001_2020', 'primary_loss_total_2001_2020', 'avg_annual_loss_ha', 'loss_per_area', 'net_change_ha', 'net_change_pct', 'recovery_gap_ha', 'disturbance_ha', 'reforestation_rate', 'deforestation_rank_value', 'annual_loss_ha_lag1', 'annual_loss_ha_lag2', 'annual_primary_loss_ha_lag1', 'annual_emissions_Mg_lag1', 'annual_loss_ha_roll3', 'annual_loss_ha_roll5'

In [12]:
# ---------------------------------------------------------
# Critical column missing-value check
# ---------------------------------------------------------
critical_cols = [
    "iso",
    "country_name",
    "continent",
    "subregion",
    "annual_loss_ha",
    "annual_primary_loss_ha",
]

existing_critical_cols = [c for c in critical_cols if c in country_year_features.columns]

print("Missing values in critical country_year_features columns:")
print(country_year_features[existing_critical_cols].isna().sum())

Missing values in critical country_year_features columns:
iso                          0
country_name               122
continent                  122
subregion                  122
annual_loss_ha               0
annual_primary_loss_ha    2551
dtype: int64


In [13]:
# ---------------------------------------------------------
# Geography quality checks
# ---------------------------------------------------------
print("Rows with unknown continent mapping:")
display(canonical_country_table[canonical_country_table["continent"] == "Unknown"])

print("Rows with unassigned subregion mapping:")
display(canonical_country_table[canonical_country_table["subregion"] == "Unassigned"])

Rows with unknown continent mapping:


,country_name,iso,continent,subregion,extent_2000_ha,area__ha,umd_tree_cover_gain__ha,loss_total_2001_2020,avg_annual_loss_ha,primary_loss_total_2001_2020,area_ha,gain_2000_2020_ha,net_change_ha,net_change_pct,recovery_gap_ha,reforestation_rate,loss_per_area,disturbance_ha,deforestation_rank_value
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned,911.242005,4.673448e+04,83.872300,76.132899,3.460586,NaN,NaN,0.0,-76.132899,-8.354850,-76.132899,0.0,NaN,<NA>,188.0
9,Antarctica,ATA,Unknown,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
49,Clipperton Island,XCL,Unknown,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
80,French Southern Territories,ATF,Unknown,Unassigned,71.636103,4.361769e+03,NaN,6.334069,6.334069,NaN,NaN,0.0,-6.334069,-8.842007,-6.334069,0.0,NaN,<NA>,202.0
116,Kosovo,XKO,Unknown,Unassigned,369937.705205,1.083659e+06,9291.695476,18081.191566,753.382982,NaN,NaN,0.0,-18081.191566,-4.887631,-18081.191566,0.0,NaN,<NA>,130.0
173,Pitcairn Islands,PCN,Unknown,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
199,Sint Maarten,SXM,Unknown,Unassigned,1022.472725,3.801754e+03,16.480192,33.472288,1.673614,6.078870,NaN,0.0,-33.472288,-3.273661,-33.472288,0.0,NaN,<NA>,193.0
220,Timor-Leste,TLS,Unknown,Unassigned,726234.903873,1.491567e+06,11134.119141,33689.274107,1403.719754,NaN,NaN,0.0,-33689.274107,-4.638895,-33689.274107,0.0,NaN,<NA>,120.0
235,United States Minor Outlying Islands,UMI,Unknown,Unassigned,145.898117,5.416798e+02,4.458699,103.868262,14.838323,50.216386,NaN,0.0,-103.868262,-71.192326,-103.868262,0.0,NaN,<NA>,182.0
239,Vatican City,VAT,Unknown,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN


Rows with unassigned subregion mapping:


,country_name,iso,continent,subregion,extent_2000_ha,area__ha,umd_tree_cover_gain__ha,loss_total_2001_2020,avg_annual_loss_ha,primary_loss_total_2001_2020,area_ha,gain_2000_2020_ha,net_change_ha,net_change_pct,recovery_gap_ha,reforestation_rate,loss_per_area,disturbance_ha,deforestation_rank_value
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned,9.112420e+02,4.673448e+04,83.872300,76.132899,3.460586,NaN,NaN,0.0,-76.132899,-8.354850,-76.132899,0.0,NaN,<NA>,188.0
9,Antarctica,ATA,Unknown,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
28,"Bonaire, Sint Eustatius and Saba",BES,North America,Unassigned,2.432087e+03,3.236599e+04,270.182869,64.232145,3.568452,23.499775,NaN,0.0,-64.232145,-2.641030,-64.232145,0.0,NaN,<NA>,190.0
33,British Indian Ocean Territory,IOT,Asia,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
48,Christmas Island,CXR,Asia,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
49,Clipperton Island,XCL,Unknown,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
50,Cocos Islands,CCK,Asia,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN
80,French Southern Territories,ATF,Unknown,Unassigned,7.163610e+01,4.361769e+03,NaN,6.334069,6.334069,NaN,NaN,0.0,-6.334069,-8.842007,-6.334069,0.0,NaN,<NA>,202.0
116,Kosovo,XKO,Unknown,Unassigned,3.699377e+05,1.083659e+06,9291.695476,18081.191566,753.382982,NaN,NaN,0.0,-18081.191566,-4.887631,-18081.191566,0.0,NaN,<NA>,130.0
173,Pitcairn Islands,PCN,Unknown,Unassigned,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.000000,NaN,NaN,<NA>,NaN


In [14]:
# ---------------------------------------------------------
# Raw source schema inspection
# Useful for debugging column mismatches
# ---------------------------------------------------------
for name, df in tables.items():
    print(f"\n{'=' * 80}")
    print(f"TABLE: {name}")
    print(f"{'=' * 80}")
    print(df.columns.tolist())
    display(df.head(2))


TABLE: iso_meta
['name', 'iso', 'continent', 'subregion']


,name,iso,continent,subregion
0,Afghanistan,AFG,Asia,Southern Asia
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned



TABLE: extent
['iso', 'umd_tree_cover_extent_2000__ha', 'area__ha']


,iso,umd_tree_cover_extent_2000__ha,area__ha
0,ABW,24.952411,1.819751e+04
1,AFG,205771.133693,6.438365e+07



TABLE: gain
['iso', 'umd_tree_cover_gain__ha']


...

,iso,umd_tree_cover_loss__year,umd_tree_cover_loss__ha,gfw_gross_emissions_co2e_all_gases__Mg
0,AFG,2001,88.092712,2.798628e+04
1,AGO,2001,101218.504554,3.929474e+07



TABLE: primary_loss_annual
['iso', 'umd_tree_cover_loss__year', 'umd_tree_cover_loss__ha', 'gfw_gross_emissions_co2e_all_gases__Mg']


,iso,umd_tree_cover_loss__year,umd_tree_cover_loss__ha,gfw_gross_emissions_co2e_all_gases__Mg
0,AGO,2001,4957.521252,3.180787e+06
1,ARG,2001,5979.954614,2.342090e+06



TABLE: deforestation_rank
['iso', 'country', 'def_per_year']


,iso,country,def_per_year
0,BRA,Brazil,1695700.0
1,IND,India,668400.0


### Visual Sanity Checks

quickly checking whether the outputs look reasonable.

The examples below visualize:

- top countries by total forest loss
- global annual forest loss trend
- relationship between annual loss and annual emissions

In [16]:
# =========================================================
# STEP 7: VALIDATION
# =========================================================

print("Canonical country table columns:")
print(canonical_country_table.columns.tolist())

print("\nCountry-year feature store columns:")
print(country_year_features.columns.tolist())

print("\nRegion aggregates columns:")
print(region_aggregates.columns.tolist())

Canonical country table columns:
['country_name', 'iso', 'continent', 'subregion', 'extent_2000_ha', 'area__ha', 'umd_tree_cover_gain__ha', 'loss_total_2001_2020', 'avg_annual_loss_ha', 'primary_loss_total_2001_2020', 'area_ha', 'gain_2000_2020_ha', 'net_change_ha', 'net_change_pct', 'recovery_gap_ha', 'reforestation_rate', 'loss_per_area', 'disturbance_ha', 'deforestation_rank_value']

Country-year feature store columns:
['iso', 'year', 'annual_loss_ha', 'annual_emissions_Mg', 'annual_primary_loss_ha', 'annual_primary_emissions_Mg', 'country_name', 'continent', 'subregion', 'area_ha', 'extent_2000_ha', 'gain_2000_2020_ha', 'loss_total_2001_2020', 'primary_loss_total_2001_2020', 'avg_annual_loss_ha', 'loss_per_area', 'net_change_ha', 'net_change_pct', 'recovery_gap_ha', 'disturbance_ha', 'reforestation_rate', 'deforestation_rank_value', 'annual_loss_ha_lag1', 'annual_loss_ha_lag2', 'annual_primary_loss_ha_lag1', 'annual_emissions_Mg_lag1', 'annual_loss_ha_roll3', 'annual_loss_ha_roll5'

In [17]:
print("Sample canonical_country_table:")
display(canonical_country_table.head())

print("\nSample country_year_features:")
display(country_year_features.head())

print("\nSample region_aggregates:")
display(region_aggregates.head(20))

Sample canonical_country_table:


,country_name,iso,continent,subregion,extent_2000_ha,area__ha,umd_tree_cover_gain__ha,loss_total_2001_2020,avg_annual_loss_ha,primary_loss_total_2001_2020,area_ha,gain_2000_2020_ha,net_change_ha,net_change_pct,recovery_gap_ha,reforestation_rate,loss_per_area,disturbance_ha,deforestation_rank_value
0,Afghanistan,AFG,Asia,Southern Asia,2.057711e+05,6.438365e+07,10738.734394,1912.066093,91.050766,NaN,NaN,0.0,-1912.066093,-0.929220,-1912.066093,0.0,NaN,<NA>,156.0
1,Akrotiri and Dhekelia,XAD,Unknown,Unassigned,9.112420e+02,4.673448e+04,83.872300,76.132899,3.460586,NaN,NaN,0.0,-76.132899,-8.354850,-76.132899,0.0,NaN,<NA>,188.0
2,Åland,ALA,Europe,Northern Europe,1.077392e+05,1.506139e+05,2582.869603,17655.780082,735.657503,NaN,NaN,0.0,-17655.780082,-16.387512,-17655.780082,0.0,NaN,<NA>,131.0
3,Albania,ALB,Europe,Southern Europe,6.484594e+05,2.872761e+06,16468.972163,47318.591026,1971.607959,NaN,NaN,0.0,-47318.591026,-7.297079,-47318.591026,0.0,NaN,<NA>,116.0
4,Algeria,DZA,Africa,Northern Africa,1.223324e+06,2.308021e+08,89145.649956,231653.346546,9652.222773,NaN,NaN,0.0,-231653.346546,-18.936383,-231653.346546,0.0,NaN,<NA>,89.0



Sample country_year_features:


,iso,year,annual_loss_ha,annual_emissions_Mg,annual_primary_loss_ha,annual_primary_emissions_Mg,country_name,continent,subregion,area_ha,...,annual_loss_ha_roll3,annual_loss_ha_roll5,annual_primary_loss_ha_roll3,annual_emissions_Mg_roll3,annual_loss_ha_yoy_change,annual_primary_loss_ha_yoy_change,annual_loss_ha_growth_rate,annual_primary_loss_ha_growth_rate,annual_loss_pct_of_extent_2000,annual_primary_loss_pct_of_extent_2000
0,ABW,2002,0.526108,289.581373,NaN,NaN,Aruba,North America,Caribbean,NaN,...,0.526108,0.526108,NaN,289.581373,NaN,NaN,NaN,NaN,2.108444,NaN
1,ABW,2003,0.977058,550.205498,NaN,NaN,Aruba,North America,Caribbean,NaN,...,0.751583,0.751583,NaN,419.893435,0.450950,NaN,0.857144,NaN,3.915684,NaN
2,ABW,2006,0.225487,44.168715,0.075162,24.575414,Aruba,North America,Caribbean,NaN,...,0.576217,0.576217,0.075162,294.651862,-0.751571,NaN,-0.769218,NaN,0.903668,0.301223
3,ABW,2009,0.375798,105.367894,0.300651,85.674361,Aruba,North America,Caribbean,NaN,...,0.526114,0.526113,0.187907,233.247369,0.150311,0.225489,0.666606,3.000018,1.506058,1.204897
4,ABW,2012,0.150325,26.254148,NaN,NaN,Aruba,North America,Caribbean,NaN,...,0.250537,0.450955,0.187907,58.596919,-0.225473,NaN,-0.599985,NaN,0.602446,NaN



Sample region_aggregates:


,aggregation_level,region_name,year,country_count,country_name_count,annual_loss_ha,annual_primary_loss_ha,annual_emissions_Mg,annual_primary_emissions_Mg,area_ha,...,annual_loss_pct_of_extent_2000_roll3,annual_loss_pct_of_extent_2000_roll5,annual_loss_pct_of_extent_2000_yoy_change,annual_loss_pct_of_extent_2000_growth_rate,annual_primary_loss_pct_of_extent_2000_lag1,annual_primary_loss_pct_of_extent_2000_lag2,annual_primary_loss_pct_of_extent_2000_roll3,annual_primary_loss_pct_of_extent_2000_roll5,annual_primary_loss_pct_of_extent_2000_yoy_change,annual_primary_loss_pct_of_extent_2000_growth_rate
0,World,Global,2001,207,201,1.335480e+07,991708.542199,4.975005e+09,6.750329e+08,0.0,...,0.335188,0.335188,NaN,NaN,NaN,NaN,2.489056e-02,2.489056e-02,NaN,NaN
1,Continent,Africa,2001,55,55,1.390064e+06,174747.875824,6.727150e+08,1.142535e+08,0.0,...,0.236206,0.236206,NaN,NaN,NaN,NaN,2.969389e-02,2.969389e-02,NaN,NaN
2,Continent,Asia,2001,40,40,1.697614e+06,188826.458164,9.164730e+08,1.619858e+08,0.0,...,0.287956,0.287956,NaN,NaN,NaN,NaN,3.202945e-02,3.202945e-02,NaN,NaN
3,Continent,Europe,2001,44,44,2.493107e+06,0.000000,4.662200e+08,0.000000e+00,0.0,...,0.259384,0.259384,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,NaN
4,Continent,North America,2001,36,36,3.689293e+06,22592.982840,1.198477e+09,1.234299e+07,0.0,...,0.463423,0.463423,NaN,NaN,NaN,NaN,2.837972e-03,2.837972e-03,NaN,NaN
5,Continent,Oceania,2001,8,8,3.095060e+05,11021.121606,7.885668e+07,9.245541e+06,0.0,...,0.299276,0.299276,NaN,NaN,NaN,NaN,1.065683e-02,1.065683e-02,NaN,NaN
6,Continent,South America,2001,13,13,3.766617e+06,593351.707521,1.637523e+09,3.764582e+08,0.0,...,0.398812,0.398812,NaN,NaN,NaN,NaN,6.282454e-02,6.282454e-02,NaN,NaN
7,Continent,Unknown,2001,11,5,8.596615e+03,1168.396244,4.740338e+06,7.468370e+05,0.0,...,0.782754,0.782754,NaN,NaN,NaN,NaN,1.063869e-01,1.063869e-01,NaN,NaN
8,Subregion,Australia and New Zealand,2001,2,2,2.693418e+05,0.075922,5.020933e+07,1.426593e+02,0.0,...,0.503092,0.503092,NaN,NaN,NaN,NaN,1.418122e-07,1.418122e-07,NaN,NaN
9,Subregion,Caribbean,2001,24,24,3.827805e+04,941.121416,1.375426e+07,4.017463e+05,0.0,...,0.387109,0.387109,NaN,NaN,NaN,NaN,9.517646e-03,9.517646e-03,NaN,NaN


In [23]:
# Critical columns to inspect
critical_cols = [
    "iso",
    "country_name",
    "continent",
    "subregion",
    "annual_loss_ha",
    "annual_primary_loss_ha",
]

existing_critical_cols = [c for c in critical_cols if c in country_year_features.columns]

print("Missing values in country_year_features critical columns:")
print(country_year_features[existing_critical_cols].isna().sum())

Missing values in country_year_features critical columns:
iso                          0
country_name               122
continent                  122
subregion                  122
annual_loss_ha               0
annual_primary_loss_ha    2551
dtype: int64


In [15]:
# ---------------------------------------------------------
# OPTIONAL VISUAL SANITY CHECKS
# ---------------------------------------------------------

# 1. Top 10 countries by cumulative forest loss
plt.figure(figsize=(10, 6))
top_loss = canonical_country_table.nlargest(10, "loss_total_2001_2020")
plt.barh(top_loss["country_name"], top_loss["loss_total_2001_2020"])
plt.gca().invert_yaxis()
plt.title("Top 10 Countries by Total Forest Loss (2001–2020)")
plt.xlabel("Total Forest Loss (ha)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

# 2. Global annual loss trend
plt.figure(figsize=(10, 6))
global_trend = region_aggregates[region_aggregates["aggregation_level"] == "World"]
plt.plot(global_trend["year"], global_trend["annual_loss_ha"])
plt.title("Global Annual Forest Loss Trend")
plt.xlabel("Year")
plt.ylabel("Annual Forest Loss (ha)")
plt.tight_layout()
plt.show()

# 3. Scatter plot: annual loss vs annual emissions
plt.figure(figsize=(10, 6))
plt.scatter(
    country_year_features["annual_loss_ha"],
    country_year_features["annual_emissions_Mg"],
    alpha=0.5
)
plt.title("Annual Forest Loss vs Annual Emissions")
plt.xlabel("Annual Forest Loss (ha)")
plt.ylabel("Annual Emissions (Mg)")
plt.tight_layout()
plt.show()

# 2. Predictive & Statistical Outputs

## Predictive and Statistical Outputs

This section extends the analytics pipeline beyond descriptive summaries into statistical and forward-looking outputs.

### New outputs introduced

1. **trend_summary**  
   A statistical summary of forest loss trajectories for:
   - countries
   - continents
   - subregions

   For each entity, the model estimates the slope of annual forest loss over time.

   - **Positive slope** → forest loss is increasing over time
   - **Negative slope** → forest loss is stabilizing or declining
   - **Near-zero slope** → relatively flat trajectory

2. **forecasts**  
   A 5-year forecast of annual forest loss for each country, based on historical trend patterns.

### Why this matters

These outputs support:

- early warning systems
- risk ranking
- future scenario analysis
- strategic intervention planning
- identification of at-risk countries and regions

### Modelling note

This implementation uses a simple linear trend model based on historical annual loss:

\[
\text{annual_loss_ha} = \alpha + \beta \cdot \text{year}
\]

Where:

- \(\alpha\) is the intercept
- \(\beta\) is the slope

The slope is used in `trend_summary`, and the fitted line is projected forward to create `forecasts`.

This is intentionally simple, interpretable, and robust enough for a first analytical iteration.

In [35]:
# =========================================================
# STEP 8: PREDICTIVE AND STATISTICAL OUTPUTS
# =========================================================

class PredictiveAnalyticsBuilder:
    """
    Build statistical trend summaries and forward forecasts from the
    country-year feature store and regional aggregates.

    Main outputs
    ------------
    1. trend_summary
       Statistical slope-based summaries for:
       - Country
       - Continent
       - Subregion

    2. forecasts
       5-year country-level forecasts for annual forest loss.

    Notes
    -----
    This first implementation uses simple linear regression over time:

        annual_loss_ha ~ year

    This gives:
    - slope
    - intercept
    - r_value
    - p_value
    - std_err

    The approach is intentionally interpretable and lightweight.
    """

    def __init__(self):
        pass

    # -----------------------------------------------------
    # Validation helpers
    # -----------------------------------------------------
    @staticmethod
    def _require_columns(df: pd.DataFrame, required_cols: List[str], df_name: str) -> None:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(
                f"Missing required column(s) {missing} in '{df_name}'. "
                f"Available columns: {list(df.columns)}"
            )

    @staticmethod
    def _safe_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
        out = df.copy()
        for col in cols:
            if col in out.columns:
                out[col] = pd.to_numeric(out[col], errors="coerce")
        return out

    @staticmethod
    def _classify_trend(slope: float, tolerance: float = 1e-9) -> str:
        """
        Classify direction of trend based on slope.
        """
        if pd.isna(slope):
            return "insufficient_data"
        if slope > tolerance:
            return "accelerating_loss"
        if slope < -tolerance:
            return "stabilizing_or_declining"
        return "flat"

    @staticmethod
    def _forecast_risk_label(avg_forecast: float, latest_actual: float) -> str:
        """
        Simple risk labelling based on forecast level relative to latest observed loss.
        """
        if pd.isna(avg_forecast):
            return "unknown"

        if pd.isna(latest_actual) or latest_actual == 0:
            if avg_forecast > 0:
                return "emerging_risk"
            return "low_risk"

        ratio = avg_forecast / latest_actual

        if ratio >= 1.25:
            return "high_risk"
        if ratio >= 1.0:
            return "watchlist"
        return "stable_or_improving"

    def _fit_linear_trend(self, series_df: pd.DataFrame, year_col: str, value_col: str) -> Dict[str, float]:
        """
        Fit a linear trend model on one time series.

        Returns
        -------
        Dict[str, float]
            slope, intercept, r_value, r_squared, p_value, std_err, n_obs
        """
        work = series_df[[year_col, value_col]].dropna().copy()

        if work.empty or len(work) < 2:
            return {
                "slope": np.nan,
                "intercept": np.nan,
                "r_value": np.nan,
                "r_squared": np.nan,
                "p_value": np.nan,
                "std_err": np.nan,
                "n_obs": len(work),
            }

        work[year_col] = pd.to_numeric(work[year_col], errors="coerce")
        work[value_col] = pd.to_numeric(work[value_col], errors="coerce")
        work = work.dropna()

        if len(work) < 2:
            return {
                "slope": np.nan,
                "intercept": np.nan,
                "r_value": np.nan,
                "r_squared": np.nan,
                "p_value": np.nan,
                "std_err": np.nan,
                "n_obs": len(work),
            }

        result = linregress(work[year_col], work[value_col])

        return {
            "slope": result.slope,
            "intercept": result.intercept,
            "r_value": result.rvalue,
            "r_squared": result.rvalue ** 2,
            "p_value": result.pvalue,
            "std_err": result.stderr,
            "n_obs": len(work),
        }

    def _build_trend_summary_for_level(
        self,
        df: pd.DataFrame,
        level_name: str,
        entity_col: str
    ) -> pd.DataFrame:
        """
        Build trend summary for one level of analysis.

        Parameters
        ----------
        df : pd.DataFrame
            Input dataframe containing yearly annual_loss_ha.
        level_name : str
            Country / Continent / Subregion
        entity_col : str
            Column identifying the entity at that level.

        Returns
        -------
        pd.DataFrame
            Trend summary table for that level.
        """
        self._require_columns(df, [entity_col, "year", "annual_loss_ha"], f"{level_name}_input")

        rows = []

        for entity, g in df.groupby(entity_col, dropna=False):
            stats = self._fit_linear_trend(g, year_col="year", value_col="annual_loss_ha")

            latest_year = pd.to_numeric(g["year"], errors="coerce").max()
            latest_actual_series = g.loc[pd.to_numeric(g["year"], errors="coerce") == latest_year, "annual_loss_ha"]
            latest_actual = latest_actual_series.mean() if not latest_actual_series.empty else np.nan

            avg_loss = pd.to_numeric(g["annual_loss_ha"], errors="coerce").mean()

            rows.append({
                "aggregation_level": level_name,
                "entity_name": entity,
                "start_year": pd.to_numeric(g["year"], errors="coerce").min(),
                "end_year": latest_year,
                "n_obs": stats["n_obs"],
                "avg_annual_loss_ha": avg_loss,
                "latest_annual_loss_ha": latest_actual,
                "loss_slope_ha_per_year": stats["slope"],
                "intercept": stats["intercept"],
                "r_value": stats["r_value"],
                "r_squared": stats["r_squared"],
                "p_value": stats["p_value"],
                "std_err": stats["std_err"],
                "trend_direction": self._classify_trend(stats["slope"]),
            })

        out = pd.DataFrame(rows)

        if not out.empty:
            out = out.sort_values(
                ["aggregation_level", "loss_slope_ha_per_year"],
                ascending=[True, False]
            ).reset_index(drop=True)

        return out

    def build_trend_summary(
        self,
        country_year_features: pd.DataFrame,
        region_aggregates: pd.DataFrame
    ) -> pd.DataFrame:
        """
        Build slope-based trend summaries for:
        - Country
        - Continent
        - Subregion
        """
        self._require_columns(
            country_year_features,
            ["iso", "country_name", "continent", "subregion", "year", "annual_loss_ha"],
            "country_year_features"
        )
        self._require_columns(
            region_aggregates,
            ["aggregation_level", "region_name", "year", "annual_loss_ha"],
            "region_aggregates"
        )

        country_df = country_year_features.copy()
        country_df = self._safe_numeric(country_df, ["year", "annual_loss_ha"])

        country_summary = self._build_trend_summary_for_level(
            df=country_df.rename(columns={"country_name": "entity_name"}),
            level_name="Country",
            entity_col="entity_name"
        )

        continent_df = region_aggregates[
            region_aggregates["aggregation_level"] == "Continent"
        ].copy()
        continent_df = self._safe_numeric(continent_df, ["year", "annual_loss_ha"])

        continent_summary = self._build_trend_summary_for_level(
            df=continent_df.rename(columns={"region_name": "entity_name"}),
            level_name="Continent",
            entity_col="entity_name"
        )

        subregion_df = region_aggregates[
            region_aggregates["aggregation_level"] == "Subregion"
        ].copy()
        subregion_df = self._safe_numeric(subregion_df, ["year", "annual_loss_ha"])

        subregion_summary = self._build_trend_summary_for_level(
            df=subregion_df.rename(columns={"region_name": "entity_name"}),
            level_name="Subregion",
            entity_col="entity_name"
        )

        trend_summary = pd.concat(
            [country_summary, continent_summary, subregion_summary],
            ignore_index=True
        )

        return trend_summary

    def build_country_forecasts(
        self,
        country_year_features: pd.DataFrame,
        forecast_horizon: int = 5
    ) -> pd.DataFrame:
        """
        Build a 5-year annual forest-loss forecast for every country.

        Parameters
        ----------
        country_year_features : pd.DataFrame
            Country-year feature store.
        forecast_horizon : int, default=5
            Number of years to project forward.

        Returns
        -------
        pd.DataFrame
            One row per country per forecast year.
        """
        self._require_columns(
            country_year_features,
            ["iso", "country_name", "continent", "subregion", "year", "annual_loss_ha"],
            "country_year_features"
        )

        df = country_year_features.copy()
        df = self._safe_numeric(df, ["year", "annual_loss_ha"])

        forecast_rows = []

        for iso, g in df.groupby("iso", dropna=False):
            work = g[["iso", "country_name", "continent", "subregion", "year", "annual_loss_ha"]].copy()
            work = work.dropna(subset=["year", "annual_loss_ha"])

            country_name = work["country_name"].dropna().iloc[0] if not work["country_name"].dropna().empty else iso
            continent = work["continent"].dropna().iloc[0] if not work["continent"].dropna().empty else "Unknown"
            subregion = work["subregion"].dropna().iloc[0] if not work["subregion"].dropna().empty else "Unassigned"

            trend_stats = self._fit_linear_trend(work, year_col="year", value_col="annual_loss_ha")

            if pd.isna(trend_stats["slope"]) or pd.isna(trend_stats["intercept"]):
                continue

            latest_year = int(work["year"].max())
            latest_actual = work.loc[work["year"] == latest_year, "annual_loss_ha"].mean()

            future_years = list(range(latest_year + 1, latest_year + forecast_horizon + 1))
            predicted_values = []

            for future_year in future_years:
                predicted_loss = trend_stats["intercept"] + trend_stats["slope"] * future_year

                # Forest loss should not go below zero
                predicted_loss = max(predicted_loss, 0)

                predicted_values.append(predicted_loss)

                forecast_rows.append({
                    "iso": iso,
                    "country_name": country_name,
                    "continent": continent,
                    "subregion": subregion,
                    "historical_end_year": latest_year,
                    "forecast_year": future_year,
                    "forecast_horizon_year": future_year - latest_year,
                    "forecast_annual_loss_ha": predicted_loss,
                    "latest_actual_annual_loss_ha": latest_actual,
                    "loss_slope_ha_per_year": trend_stats["slope"],
                    "r_squared": trend_stats["r_squared"],
                    "trend_direction": self._classify_trend(trend_stats["slope"]),
                })

        forecasts = pd.DataFrame(forecast_rows)

        if forecasts.empty:
            return forecasts

        # Country-level summary indicators repeated across forecast rows
        country_risk = (
            forecasts.groupby(["iso", "country_name", "continent", "subregion"], as_index=False)
            .agg(
                avg_forecast_annual_loss_ha=("forecast_annual_loss_ha", "mean"),
                max_forecast_annual_loss_ha=("forecast_annual_loss_ha", "max"),
                latest_actual_annual_loss_ha=("latest_actual_annual_loss_ha", "mean"),
                loss_slope_ha_per_year=("loss_slope_ha_per_year", "mean"),
                r_squared=("r_squared", "mean"),
            )
        )

        country_risk["forecast_risk_label"] = country_risk.apply(
            lambda row: self._forecast_risk_label(
                avg_forecast=row["avg_forecast_annual_loss_ha"],
                latest_actual=row["latest_actual_annual_loss_ha"]
            ),
            axis=1
        )

        forecasts = forecasts.merge(
            country_risk,
            on=[
                "iso", "country_name", "continent", "subregion",
                "latest_actual_annual_loss_ha", "loss_slope_ha_per_year", "r_squared"
            ],
            how="left"
        )

        forecasts = forecasts.sort_values(
            ["forecast_year", "forecast_annual_loss_ha"],
            ascending=[True, False]
        ).reset_index(drop=True)

        return forecasts

## Build Predictive Outputs

This section runs the predictive builder and creates two new analytical assets:

1. `trend_summary`
2. `forecasts`

These outputs extend the pipeline from descriptive analytics into statistical interpretation and forward-looking risk analysis.

In [36]:
# =========================================================
# STEP 8: BUILD PREDICTIVE OUTPUTS
# =========================================================

pab = PredictiveAnalyticsBuilder()

trend_summary = pab.build_trend_summary(
    country_year_features=country_year_features,
    region_aggregates=region_aggregates
)

forecasts = pab.build_country_forecasts(
    country_year_features=country_year_features,
    forecast_horizon=5
)

print("trend_summary shape:", trend_summary.shape)
print("forecasts shape:", forecasts.shape)

trend_summary shape: (240, 14)
forecasts shape: (1060, 15)


## Validation and Inspection for Predictive Outputs

This section validates the statistical and forecast outputs to ensure:

- slope estimates were created
- countries were assigned meaningful trend labels
- forecast rows were created for each future year
- high-risk countries can be identified

In [37]:
# ---------------------------------------------------------
# TREND SUMMARY INSPECTION
# ---------------------------------------------------------
print("Trend summary columns:")
print(trend_summary.columns.tolist())

print("\nSample trend summary:")
display(trend_summary.head(20))

print("\nTrend direction counts:")
display(trend_summary["trend_direction"].value_counts(dropna=False))

Trend summary columns:
['aggregation_level', 'entity_name', 'start_year', 'end_year', 'n_obs', 'avg_annual_loss_ha', 'latest_annual_loss_ha', 'loss_slope_ha_per_year', 'intercept', 'r_value', 'r_squared', 'p_value', 'std_err', 'trend_direction']

Sample trend summary:


,aggregation_level,entity_name,start_year,end_year,n_obs,avg_annual_loss_ha,latest_annual_loss_ha,loss_slope_ha_per_year,intercept,r_value,r_squared,p_value,std_err,trend_direction
0,Country,Russia,2001,2024,24,3.701295e+06,5.180133e+06,141275.036798,-2.806147e+08,0.686887,0.471814,2.094710e-04,31868.561752,accelerating_loss
1,Country,Canada,2001,2024,24,2.610311e+06,5.165897e+06,93241.450462,-1.850381e+08,0.447301,0.200078,2.840942e-02,39748.611693,accelerating_loss
2,Country,Democratic Republic of the Congo,2001,2024,24,8.780145e+05,1.377807e+06,52771.981411,-1.053256e+08,0.891388,0.794572,5.133548e-09,5720.781436,accelerating_loss
3,Country,Bolivia,2001,2024,24,4.074446e+05,1.813613e+06,33306.375439,-6.662164e+07,0.667517,0.445579,3.658133e-04,7920.887446,accelerating_loss
4,Country,Brazil,2001,2024,24,3.054878e+06,4.391694e+06,21994.383437,-4.120882e+07,0.184056,0.033877,3.892700e-01,25041.846555,accelerating_loss
5,Country,Australia,2001,2024,24,3.840972e+05,1.806699e+05,20392.308726,-4.065542e+07,0.278790,0.077724,1.870975e-01,14976.405210,accelerating_loss
6,Country,Laos,2001,2024,24,2.151302e+05,3.512260e+05,16758.588117,-3.351153e+07,0.939278,0.882244,1.075070e-11,1305.339998,accelerating_loss
7,Country,Myanmar,2001,2024,24,2.144708e+05,2.769778e+05,12466.007299,-2.487337e+07,0.879586,0.773672,1.508312e-08,1437.497741,accelerating_loss
8,Country,Madagascar,2001,2024,24,2.147140e+05,2.312187e+05,11892.253744,-2.371845e+07,0.685666,0.470138,2.172291e-04,2691.670768,accelerating_loss
9,Country,Angola,2001,2024,24,1.762596e+05,2.833404e+05,10114.595889,-2.017936e+07,0.928247,0.861642,6.401961e-11,864.123351,accelerating_loss



Trend direction counts:


trend_direction
accelerating_loss           168
stabilizing_or_declining     67
insufficient_data             4
flat                          1
Name: count, dtype: int64

In [38]:
# ---------------------------------------------------------
# FORECAST INSPECTION
# ---------------------------------------------------------
print("Forecast columns:")
print(forecasts.columns.tolist())

print("\nSample forecasts:")
display(forecasts.head(20))

print("\nForecast risk label counts:")
display(forecasts["forecast_risk_label"].value_counts(dropna=False))

Forecast columns:
['iso', 'country_name', 'continent', 'subregion', 'historical_end_year', 'forecast_year', 'forecast_horizon_year', 'forecast_annual_loss_ha', 'latest_actual_annual_loss_ha', 'loss_slope_ha_per_year', 'r_squared', 'trend_direction', 'avg_forecast_annual_loss_ha', 'max_forecast_annual_loss_ha', 'forecast_risk_label']

Sample forecasts:


,iso,country_name,continent,subregion,historical_end_year,forecast_year,forecast_horizon_year,forecast_annual_loss_ha,latest_actual_annual_loss_ha,loss_slope_ha_per_year,r_squared,trend_direction,avg_forecast_annual_loss_ha,max_forecast_annual_loss_ha,forecast_risk_label
0,UMI,United States Minor Outlying Islands,Unknown,Unassigned,2012,2013,1,7.994353,10.452939,-1.064618,0.021609,stabilizing_or_declining,5.865118,7.994353,stable_or_improving
1,UMI,United States Minor Outlying Islands,Unknown,Unassigned,2012,2014,2,6.929735,10.452939,-1.064618,0.021609,stabilizing_or_declining,5.865118,7.994353,stable_or_improving
2,SAU,Saudi Arabia,Asia,Western Asia,2013,2014,1,0.000000,0.000000,0.000000,NaN,flat,0.000000,0.000000,low_risk
3,MUS,Mauritius,Africa,Eastern Africa,2014,2015,1,326.753853,531.751975,14.496219,0.184256,accelerating_loss,355.746291,384.738729,stable_or_improving
4,UMI,United States Minor Outlying Islands,Unknown,Unassigned,2012,2015,3,5.865118,10.452939,-1.064618,0.021609,stabilizing_or_declining,5.865118,7.994353,stable_or_improving
5,BMU,Bermuda,North America,Northern America,2014,2015,1,0.851288,0.848839,0.045437,0.243591,accelerating_loss,NaN,NaN,NaN
6,SAU,Saudi Arabia,Asia,Western Asia,2013,2015,2,0.000000,0.000000,0.000000,NaN,flat,0.000000,0.000000,low_risk
7,MUS,Mauritius,Africa,Eastern Africa,2014,2016,2,341.250072,531.751975,14.496219,0.184256,accelerating_loss,355.746291,384.738729,stable_or_improving
8,UMI,United States Minor Outlying Islands,Unknown,Unassigned,2012,2016,4,4.800500,10.452939,-1.064618,0.021609,stabilizing_or_declining,5.865118,7.994353,stable_or_improving
9,BMU,Bermuda,North America,Northern America,2014,2016,2,0.896725,0.848839,0.045437,0.243591,accelerating_loss,NaN,NaN,NaN



Forecast risk label counts:


forecast_risk_label
NaN                    340
high_risk              290
stable_or_improving    245
watchlist              160
low_risk                20
emerging_risk            5
Name: count, dtype: int64

In [46]:
# ---------------------------------------------------------
# TOP POSITIVE SLOPES
# ---------------------------------------------------------

print("Top countries with strongest positive forest-loss slope:")
display(
    trend_summary[trend_summary["aggregation_level"] == "Country"]
    .sort_values("loss_slope_ha_per_year", ascending=False)
    .head(15)
)

print("\nTop continents with strongest positive forest-loss slope:")
display(
    trend_summary[trend_summary["aggregation_level"] == "Continent"]
    .sort_values("loss_slope_ha_per_year", ascending=False)
    .head(10)
)

print("\nTop subregions with strongest positive forest-loss slope:")
display(
    trend_summary[trend_summary["aggregation_level"] == "Subregion"]
    .sort_values("loss_slope_ha_per_year", ascending=False)
    .head(15)
)

Top countries with strongest positive forest-loss slope:


,aggregation_level,entity_name,start_year,end_year,n_obs,avg_annual_loss_ha,latest_annual_loss_ha,loss_slope_ha_per_year,intercept,r_value,r_squared,p_value,std_err,trend_direction
0,Country,Russia,2001,2024,24,3.701295e+06,5.180133e+06,141275.036798,-2.806147e+08,0.686887,0.471814,2.094710e-04,31868.561752,accelerating_loss
1,Country,Canada,2001,2024,24,2.610311e+06,5.165897e+06,93241.450462,-1.850381e+08,0.447301,0.200078,2.840942e-02,39748.611693,accelerating_loss
2,Country,Democratic Republic of the Congo,2001,2024,24,8.780145e+05,1.377807e+06,52771.981411,-1.053256e+08,0.891388,0.794572,5.133548e-09,5720.781436,accelerating_loss
3,Country,Bolivia,2001,2024,24,4.074446e+05,1.813613e+06,33306.375439,-6.662164e+07,0.667517,0.445579,3.658133e-04,7920.887446,accelerating_loss
4,Country,Brazil,2001,2024,24,3.054878e+06,4.391694e+06,21994.383437,-4.120882e+07,0.184056,0.033877,3.892700e-01,25041.846555,accelerating_loss
5,Country,Australia,2001,2024,24,3.840972e+05,1.806699e+05,20392.308726,-4.065542e+07,0.278790,0.077724,1.870975e-01,14976.405210,accelerating_loss
6,Country,Laos,2001,2024,24,2.151302e+05,3.512260e+05,16758.588117,-3.351153e+07,0.939278,0.882244,1.075070e-11,1305.339998,accelerating_loss
7,Country,Myanmar,2001,2024,24,2.144708e+05,2.769778e+05,12466.007299,-2.487337e+07,0.879586,0.773672,1.508312e-08,1437.497741,accelerating_loss
8,Country,Madagascar,2001,2024,24,2.147140e+05,2.312187e+05,11892.253744,-2.371845e+07,0.685666,0.470138,2.172291e-04,2691.670768,accelerating_loss
9,Country,Angola,2001,2024,24,1.762596e+05,2.833404e+05,10114.595889,-2.017936e+07,0.928247,0.861642,6.401961e-11,864.123351,accelerating_loss



Top continents with strongest positive forest-loss slope:


,aggregation_level,entity_name,start_year,end_year,n_obs,avg_annual_loss_ha,latest_annual_loss_ha,loss_slope_ha_per_year,intercept,r_value,r_squared,p_value,std_err,trend_direction
211,Continent,Europe,2001,2024,24,4.888343e+06,6.631575e+06,182577.878651,-3.625496e+08,0.761042,0.579185,1.574163e-05,33179.810005,accelerating_loss
212,Continent,Africa,2001,2024,24,2.746127e+06,4.082933e+06,156441.767515,-3.120929e+08,0.887992,0.788529,7.086009e-09,17272.590113,accelerating_loss
213,Continent,North America,2001,2024,24,5.210445e+06,7.565035e+06,89209.471067,-1.743236e+08,0.442244,0.195580,3.047395e-02,38572.612805,accelerating_loss
214,Continent,Asia,2001,2024,24,3.387554e+06,3.417752e+06,67946.442069,-1.333547e+08,0.511276,0.261403,1.066591e-02,24350.248307,accelerating_loss
215,Continent,South America,2001,2024,24,4.768841e+06,7.567844e+06,64945.711080,-1.259344e+08,0.414827,0.172081,4.384177e-02,30371.499247,accelerating_loss
216,Continent,Oceania,2001,2024,24,5.425713e+05,3.166175e+05,24842.877218,-4.945372e+07,0.333962,0.111531,1.107314e-01,14949.095266,accelerating_loss
217,Continent,Unknown,2001,2024,24,1.090757e+04,1.312828e+04,133.471682,-2.577042e+05,0.341018,0.116293,1.029400e-01,78.443156,accelerating_loss



Top subregions with strongest positive forest-loss slope:


,aggregation_level,entity_name,start_year,end_year,n_obs,avg_annual_loss_ha,latest_annual_loss_ha,loss_slope_ha_per_year,intercept,r_value,r_squared,p_value,std_err,trend_direction
218,Subregion,Eastern Europe,2001,2024,24,3.919832e+06,5.450356e+06,149675.368050,-2.973018e+08,0.703345,0.494694,1.261175e-04,32251.380321,accelerating_loss
219,Subregion,Middle Africa,2001,2024,24,1.272934e+06,2.065687e+06,77133.100290,-1.539574e+08,0.912269,0.832235,5.417513e-10,7383.406465,accelerating_loss
220,Subregion,Northern America,2001,2024,24,4.671265e+06,6.762437e+06,76650.867735,-1.495886e+08,0.390364,0.152384,5.930767e-02,38542.121709,accelerating_loss
221,Subregion,South America,2001,2024,24,4.768841e+06,7.567844e+06,64945.711080,-1.259344e+08,0.414827,0.172081,4.384177e-02,30371.499247,accelerating_loss
222,Subregion,South-eastern Asia,2001,2024,24,2.610281e+06,2.454913e+06,50905.820878,-9.983768e+07,0.449805,0.202325,2.742911e-02,21549.862193,accelerating_loss
223,Subregion,Western Africa,2001,2024,24,6.050942e+05,8.394811e+05,43385.252357,-8.670773e+07,0.796933,0.635101,3.150543e-06,7011.247452,accelerating_loss
224,Subregion,Eastern Africa,2001,2024,24,7.811730e+05,1.128193e+06,36753.191351,-7.318462e+07,0.875449,0.766410,2.143848e-08,4325.935805,accelerating_loss
225,Subregion,Australia and New Zealand,2001,2024,24,4.468105e+05,2.201665e+05,20671.184123,-4.115395e+07,0.281901,0.079468,1.820218e-01,14999.529184,accelerating_loss
226,Subregion,Northern Europe,2001,2024,24,6.215386e+05,7.639434e+05,19139.372916,-3.789645e+07,0.822914,0.677188,7.954653e-07,2817.322745,accelerating_loss
227,Subregion,Central America,2001,2024,24,4.926178e+05,7.712843e+05,11814.674830,-2.328442e+07,0.547093,0.299311,5.662542e-03,3854.002356,accelerating_loss


In [47]:
# ---------------------------------------------------------
# HIGH-RISK FORECAST COUNTRIES
# ---------------------------------------------------------

country_forecast_summary = (
    forecasts.groupby(["iso", "country_name", "continent", "subregion"], as_index=False)
    .agg(
        latest_actual_annual_loss_ha=("latest_actual_annual_loss_ha", "mean"),
        avg_forecast_annual_loss_ha=("avg_forecast_annual_loss_ha", "mean"),
        max_forecast_annual_loss_ha=("max_forecast_annual_loss_ha", "mean"),
        loss_slope_ha_per_year=("loss_slope_ha_per_year", "mean"),
        r_squared=("r_squared", "mean"),
        forecast_risk_label=("forecast_risk_label", "first"),
    )
)

print("Countries flagged as high risk:")
display(
    country_forecast_summary[country_forecast_summary["forecast_risk_label"] == "high_risk"]
    .sort_values("avg_forecast_annual_loss_ha", ascending=False)
    .head(20)
)

Countries flagged as high risk:


,iso,country_name,continent,subregion,latest_actual_annual_loss_ha,avg_forecast_annual_loss_ha,max_forecast_annual_loss_ha,loss_slope_ha_per_year,r_squared,forecast_risk_label
84,IDN,Indonesia,Asia,South-eastern Asia,1.120264e+06,1.426964e+06,1.440090e+06,6563.279242,0.010116,high_risk
10,AUS,Australia,Oceania,Australia and New Zealand,1.806699e+05,6.797857e+05,7.205703e+05,20392.308726,0.077724,high_risk
102,LAO,Laos,Asia,South-eastern Asia,3.512260e+05,4.581297e+05,4.916469e+05,16758.588117,0.882244,high_risk
122,MMR,Myanmar,Asia,South-eastern Asia,2.769778e+05,3.952279e+05,4.201599e+05,12466.007299,0.773672,high_risk
131,MYS,Malaysia,Asia,South-eastern Asia,2.825336e+05,3.917083e+05,3.923552e+05,-323.438741,0.000339,high_risk
37,CIV,Côte d'Ivoire,Africa,Western Africa,1.651197e+05,2.730479e+05,2.878035e+05,7377.801666,0.365795,high_risk
104,LBR,Liberia,Africa,Western Africa,1.664990e+05,2.306856e+05,2.480097e+05,8662.059477,0.696094,high_risk
70,GIN,Guinea,Africa,Western Africa,1.468016e+05,2.294999e+05,2.479800e+05,9240.024107,0.675548,high_risk
164,SLE,Sierra Leone,Africa,Western Africa,1.306691e+05,2.093651e+05,2.257634e+05,8199.116004,0.523354,high_risk
99,KHM,Cambodia,Asia,South-eastern Asia,9.347786e+04,1.683309e+05,1.747587e+05,3213.898143,0.184893,high_risk


In [48]:
# ---------------------------------------------------------
# VISUALIZE HISTORICAL + FORECAST FOR ONE COUNTRY
# ---------------------------------------------------------

selected_country = country_year_features["country_name"].dropna().iloc[0]

hist = country_year_features[
    country_year_features["country_name"] == selected_country
][["year", "annual_loss_ha"]].copy()

fc = forecasts[
    forecasts["country_name"] == selected_country
][["forecast_year", "forecast_annual_loss_ha"]].copy()

plt.figure(figsize=(10, 6))
plt.plot(hist["year"], hist["annual_loss_ha"], marker="o", label="Historical")
plt.plot(fc["forecast_year"], fc["forecast_annual_loss_ha"], marker="o", linestyle="--", label="Forecast")
plt.title(f"Historical and Forecast Annual Forest Loss: {selected_country}")
plt.xlabel("Year")
plt.ylabel("Annual Forest Loss (ha)")
plt.legend()
plt.tight_layout()
plt.show()